# Set up

In [14]:
# Import libraries
import pandas as pd
import numpy as np
import re, os, sqlite3
from datetime import datetime

# Function to extract score from options based on values in the response
def val_score_mapping(s1,s2):
    
    split_options = s2.strip().split("),")
    split_response = s1.strip().split(": ")[1].split(',')
    scores = {} 

    for i in split_options:
        if len(i) > 0:
            val_num = i.split("(score")[0].split(': ')[1].strip()
            score_num = i.split("(score")[1].split(': ')[1].strip()
            scores[val_num] = score_num

    response_score_mapping = {split_response[i].strip(): scores[split_response[i].strip()] for i in range(len(split_response))}
    list_response_score_mapping = list(response_score_mapping.values())
    str_response_score_mapping = ', '.join(str(value) for value in list_response_score_mapping)
    return str_response_score_mapping.replace(')', '')

# Function to cleanup and split time range in the response
def clean_time_range(df, column_name):
    cleaned = [] 
    for i in range(len(df[column_name])):
        if pd.notna(df[column_name][i]) and str(df[column_name][i]).startswith('time_range'):
            t = re.sub(r'[a-zA-Z\s+(\)_:]', '', df[column_name][i])
            t = t.replace(',', ':')
            if re.search(r'^[0-9]:', t): #9,30/12,30
                ttemp = '0' + t
            elif re.search(r':[0-9]$', t): #12,5/12,30
                ttemp = t.replace(':', ':0')
            else:
                ttemp = t
            # tpos = datetime.strptime(str(ttemp), '%H:%M')
            # thm = tpos.strftime('%H:%M')
            thm = ttemp      
        elif pd.notna(df[column_name][i]) and str(df[column_name][i]).startswith(' to'):
            t = re.sub(r'[a-zA-Z\s+(\)_:]', '', df[column_name][i])
            t = t.replace(',', ':')
            if re.search(r'^[0-9]:', t): #9,30/12,30
                ttemp = '0' + t
            elif re.search(r':[0-9]$', t): #12,5/12,30
                ttemp = t.replace(':', ':0')
            else:
                ttemp = t
            thm = ttemp      
        else:
            ttemp = df[column_name][i] 
            thm = ttemp
        cleaned.append(thm)
    return cleaned

#### Read in data

In [16]:
# Set up path variable and get list of responses.csv files

input_path = os.path.expanduser('~/NIMH EMA Data/Input Files/') #home directory > replace with your EMA folder > new folder called Input Files
all_files = os.listdir(os.path.join(input_path, 'EMA_applet_data'))
files = [file for file in all_files if file.startswith('responses')]
input_files = os.listdir(input_path)
output_path = os.path.expanduser('~/NIMH EMA Data/Output Files/')

# Read all the responses.csv files
responses_all = []
for i in range(len(files)):
    temp_df = pd.read_csv(os.path.join(input_path, 'EMA_applet_data', files[i]), encoding='ISO-8859-1')
    responses_all.append(temp_df)

# Concat responses.csv to one file
dat_full = pd.concat(responses_all, ignore_index=True)

dat_full = dat_full.map(str)

# Write out the concatenated responses.csv file
dat_full.to_csv(os.path.join(output_path,'responses_all.csv'),index=False)

# Data Cleaning

In [17]:
# Rename target_id column as it contains special characters
dat_full.rename(columns={'ï»¿target_id': 'target_id'}, inplace=True) 

# Add utc_timezone_offset
dat_full['offset'] = np.where(dat_full['utc_timezone_offset']=='nan', 0, 1)

dat_full['activity_start_time_offsetADD'] = np.where(dat_full['utc_timezone_offset'] != 'nan', pd.to_numeric(dat_full['activity_start_time'], errors='coerce')+ (pd.to_numeric(dat_full['utc_timezone_offset'], errors='coerce')* 60 * 1000) , pd.to_numeric(dat_full['activity_start_time'], errors='coerce'))
dat_full['activity_end_time_offsetADD'] = np.where(dat_full['utc_timezone_offset'] != 'nan', pd.to_numeric(dat_full['activity_end_time'], errors='coerce')+ (pd.to_numeric(dat_full['utc_timezone_offset'], errors='coerce')* 60 * 1000) , pd.to_numeric(dat_full['activity_end_time'], errors='coerce'))
dat_full['activity_schedule_start_time_offsetADD'] = np.where(dat_full['utc_timezone_offset'] != 'nan', pd.to_numeric(dat_full['activity_schedule_start_time'], errors='coerce')+(pd.to_numeric(dat_full['utc_timezone_offset'], errors='coerce')* 60 * 1000) , pd.to_numeric(dat_full['activity_schedule_start_time'], errors='coerce'))

dat_full['activity_start_time_offsetADD'] = pd.to_numeric(dat_full['activity_start_time_offsetADD'], downcast='integer')
dat_full['activity_end_time_offsetADD'] = pd.to_numeric(dat_full['activity_end_time_offsetADD'], downcast='integer')
dat_full['activity_schedule_start_time_offsetADD'] = pd.to_numeric(dat_full['activity_schedule_start_time_offsetADD'], downcast='integer')

dat_full['start_Time'] = dat_full['activity_start_time_offsetADD']
dat_full['end_Time'] = dat_full['activity_end_time_offsetADD']
dat_full['schedule_Time'] = dat_full['activity_schedule_start_time_offsetADD']

In [18]:
# Adjusting start and end times
dat_full = dat_full.groupby(['secret_user_id', 'activity_flow_id', 'activity_schedule_start_time'], group_keys=True).apply(lambda x: x.assign(start_Time=x['start_Time'].min(), end_Time=x['end_Time'].max())).reset_index(drop=True)

/var/folders/gr/8ff1xlg55jgfk7328g8nqgyh0000gv/T/ipykernel_62035/2339974581.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  dat_full = dat_full.groupby(['secret_user_id', 'activity_flow_id', 'activity_schedule_start_time'], group_keys=True).apply(lambda x: x.assign(start_Time=x['start_Time'].min(), end_Time=x['end_Time'].max())).reset_index(drop=True)


In [19]:
# Employ cleaning code provided by Curious team

dat_subset = dat_full[['activity_submission_id', 'activity_schedule_start_time', 'secret_user_id', 'userId',
       'activity_id', 'activity_name', 'item_name', 'item_response', 'item_response_options',
       'activity_schedule_id', 'start_Time', 'end_Time', 'schedule_Time', 'applet_version', 'activity_start_time', 'offset']]

# Create additional column to add scores in the next steps
dat_subset = dat_subset.copy()
dat_subset['item_response_scores'] = None

# Cleanup
for i in range(len(dat_subset['item_response'])):
    if re.search(r'score: ', dat_subset['item_response_options'][i]):
        s = val_score_mapping(dat_subset['item_response'][i], dat_subset['item_response_options'][i])
        dat_subset.loc[i, 'item_response_scores'] = s  
    if re.search(r'value', dat_subset['item_response'][i]):
        r = dat_subset['item_response'][i].replace("value: ", "")
        dat_subset.loc[i, 'item_response'] = r
    elif re.search(r'time:', dat_subset['item_response'][i]):
        if re.search(r'hr [0-9],', dat_subset['item_response'][i]): 
            egapp = dat_subset['item_response'][i]. replace('time: hr ', '0')
            if re.search(r', min [0-9]$', egapp): 
                egtemp = egapp.replace(', min ', ':0')
            elif re.search(r', min [0-9][0-9]$', egapp): 
                egtemp = egapp.replace(', min ', ':')
            egpos = datetime.strptime(egtemp, '%H:%M')
            egpos2 = egpos.strftime('%H:%M')
            dat_subset.loc[i, 'item_response'] = egpos2
        elif re.search(r'hr [0-9][0-9],', dat_subset['item_response'][i]):
            egapp = dat_subset['item_response'][i].replace('time: hr ', '')
            if re.search(r', min [0-9]$', egapp): 
                egtemp = egapp.replace(', min ', ':0')
            elif re.search(r', min [0-9][0-9]$', egapp): 
                egtemp = egapp.replace(', min ', ':')
            egpos = datetime.strptime(egtemp, '%H:%M')
            egpos2 = egpos.strftime('%H:%M')
            dat_subset.loc[i, 'item_response'] = egpos2
    elif re.search(r'geo:', dat_subset['item_response'][i]):
        g = dat_subset['item_response'][i].replace('geo: ', '')
        dat_subset.loc[i, 'item_response'] = g

# Combine scores and other formats of responses into one column
dat_subset['item_response2'] = np.where(dat_subset['item_response_scores'].isna(), dat_subset['item_response'], dat_subset['item_response_scores'])

# Sort and Select required columns
dat_subset = dat_subset.sort_values(by=['secret_user_id', 'activity_id', 'activity_name', 'schedule_Time', 'activity_start_time'])

### Additional Cleaning

In [20]:
dat_subset = dat_subset[['userId', 'secret_user_id', 'activity_id', 'activity_name', 'offset', 'item_name', 'item_response', 
                         'item_response_scores', 'item_response2','item_response_options', 'start_Time',
                         'end_Time', 'schedule_Time', 'activity_schedule_start_time', 'applet_version', 'activity_submission_id', 'activity_schedule_id']]
 
# Make sure there are no NAs 
dat_subset['schedule_Time'] = np.where(dat_subset['schedule_Time'].isna(), 'NO SCHEDULE', dat_subset['schedule_Time'])

### Widening Data

In [21]:
# Add answer_ids to help with debugging in the final output
answers = dat_subset.groupby(['userId', 'secret_user_id', 'activity_name', 'activity_schedule_id', 'start_Time', 'end_Time', 'schedule_Time', 'offset', 'applet_version'])['activity_submission_id'].apply(lambda x: '|'.join(x.astype(str))).reset_index()

# Widen data
dat_wide = pd.pivot_table(dat_subset, index=['userId', 'secret_user_id', 'activity_name', 'activity_schedule_id', 'start_Time', 'end_Time', 'schedule_Time', 'offset', 'applet_version'], columns='item_name', values='item_response2', aggfunc='last').reset_index()

# Join Wide format table with answers to include concatenated answer_ids
dat_wide = pd.merge(dat_wide, answers, on=['userId', 'secret_user_id', 'activity_name', 'activity_schedule_id', 'start_Time', 'end_Time', 'schedule_Time', 'offset', 'applet_version'], how='outer')

dat_wide.head()

,userId,secret_user_id,activity_name,activity_schedule_id,start_Time,end_Time,schedule_Time,offset,applet_version,event_category,...,now_where,pain_intensity,since_eaten_amount,since_had_drink,since_internet,since_pain,since_pain_where,since_physical_activity,since_times_eat,activity_submission_id
0,a4314838-bb7f-49d1-9617-b0bdff307e36,9999-999,Afternoon Assessment,107ba4a9-4882-424f-a4ef-5e6d53d690e3,1774627985352,1774628080267,1774626300000,1,1.1.0,2,...,0,5,3,3,1,1,2,3,1,543a00d7-51f5-4dc4-9a8f-a6f422ef82d5|543a00d7-...
1,a4314838-bb7f-49d1-9617-b0bdff307e36,9999-999,Morning Assessment,0c57e354-2662-48bb-8620-d11f6d8ef6d2,1774626457202,1774626518198,1774625400000,1,1.1.0,3,...,0,NaN,4,1,3,0,NaN,"3, 4",1,deb446fe-0d58-4cef-8c62-b4b89592ee94|deb446fe-...


### Specific item cleaning

In [22]:
# Use the function from the Set Up section to split headache_time_range column into start and end times
headache_times = ['headache_time_start', 'headache_time_end']

for i in headache_times: 
    if i in dat_wide.columns:
        dat_wide[i] = clean_time_range(dat_wide, i)
        dat_wide[i] = clean_time_range(dat_wide, i)

### Timestamp cleaning

In [23]:
dat_wide_full = dat_wide.map(str)

# Epoch to Timestamp
dat_wide_full['start_Time'] = pd.to_numeric(dat_wide_full['start_Time'], errors='coerce')
dat_wide_full['start_Time'] = pd.to_datetime(dat_wide_full['start_Time'] / 1000, unit='s')

dat_wide_full['end_Time'] = pd.to_numeric(dat_wide_full['end_Time'], errors='coerce')
dat_wide_full['end_Time'] = pd.to_datetime(dat_wide_full['end_Time'] / 1000, unit='s')

dat_wide_full['schedule_Time'] = pd.to_numeric(dat_wide_full['schedule_Time'], errors='coerce')
dat_wide_full['schedule_Time'] = pd.to_datetime(dat_wide_full['schedule_Time'] / 1000, unit='s')

# Timestamp cleanup 
dat_wide_full['schedule_Time'] = pd.to_datetime(dat_wide_full['schedule_Time'])
dat_wide_full['schedule_Time'] = np.where(dat_wide_full['offset']== '1', dat_wide_full['schedule_Time'],  dat_wide_full['schedule_Time'].dt.tz_localize('UTC').dt.tz_convert('America/New_York').dt.tz_localize(None)) 

dat_wide_full['start_Time'] = pd.to_datetime(dat_wide_full['start_Time'])
dat_wide_full['start_Time'] = dat_wide_full['start_Time'].dt.floor('1s')
dat_wide_full['start_Time'] = np.where(dat_wide_full['offset']== '1', dat_wide_full['start_Time'], dat_wide_full['start_Time'].dt.tz_localize('UTC').dt.tz_convert('America/New_York').dt.tz_localize(None))

dat_wide_full['end_Time'] = pd.to_datetime(dat_wide_full['end_Time'])
dat_wide_full['end_Time'] = dat_wide_full['end_Time'].dt.floor('1s')
dat_wide_full['end_Time'] = np.where(dat_wide_full['offset']== '1', dat_wide_full['end_Time'], dat_wide_full['end_Time'].dt.tz_localize('UTC').dt.tz_convert('America/New_York').dt.tz_localize(None))

# Creating Dates
dat_wide_full['scheduled_Date'] = dat_wide_full['schedule_Time'].dt.date
dat_wide_full['start_Date'] = dat_wide_full['start_Time'].dt.date

In [24]:
# Timestamp formatting - TO MAKE SURE!
dat_wide_full['schedule_Time'] = pd.to_datetime(dat_wide_full['schedule_Time'], format='mixed')
dat_wide_full['start_Time'] = pd.to_datetime(dat_wide_full['start_Time'], format="mixed")
dat_wide_full['end_Time'] = pd.to_datetime(dat_wide_full['end_Time'])

final = dat_wide_full

### Flows Final

In [25]:
#################################################################### PLEASE SELECT APPROPRIATE ITEMS & ORDER ###############################################################

# Specify the required columns, in desired order, for the final output

cols = [
    'userId',
    'activity_schedule_id',
    'activity_submission_id',
    'secret_user_id',
     'activity_name',
     'schedule_Time',
     'start_Time',
     'end_Time',
     'applet_version',
    'morning_sleep_quantity',
    'morning_bedtime',
    'morning_waketime',
    'morning_sleep_quality',
    'morning_sleep_problems',
    'now_where',
    'now_inside',
    'now_outside',
    'now_company',
    'now_activity',
    'since_internet',
    'now_sadness',
    'now_anxiousness',
    'now_active',
    'now_tired',
    'now_thoughts_positive',
    'now_thoughts_negative',
    'instructions', #event_instructions
    'event_emotion',
    'event_category',
    'event_stress',
    'since_physical_activity',
    'since_had_drink',
    'not_alcohol_amount',
    'since_times_eat',
    'since_eaten_amount',
    'since_pain',
    'since_pain_where',
    'pain_intensity',
    'day_physical_health',
    'day_problem_categories',
    'day_bother'
    ]

# Select and arrange columns, adding manual cols as NA if needed (to ensure all variables are displayed)
final = final.reindex(columns=cols)

# Sort by user ID and time of assessment
final = final.sort_values(by=['secret_user_id', 'schedule_Time', 'start_Time'])

In [26]:
# Ensure consistent NA wording across all columns
final_out = final.copy()
final_out.replace('nan', 'NA', inplace=True)
final_out.fillna('NA', inplace=True)
final_out.replace('NA NA', 'NA', inplace=True)
final_out.rename(columns = {'userId_x':'user_id'},inplace=True)

# Exclude test flows
final_out = final_out[final_out['activity_name'] != 'Test Flow (All Activities)']

# Check final flow data
final_out.head()

/var/folders/gr/8ff1xlg55jgfk7328g8nqgyh0000gv/T/ipykernel_62035/2126822779.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'NA' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  final_out.fillna('NA', inplace=True)


,userId,activity_schedule_id,activity_submission_id,secret_user_id,activity_name,schedule_Time,start_Time,end_Time,applet_version,morning_sleep_quantity,...,since_had_drink,not_alcohol_amount,since_times_eat,since_eaten_amount,since_pain,since_pain_where,pain_intensity,day_physical_health,day_problem_categories,day_bother
1,a4314838-bb7f-49d1-9617-b0bdff307e36,0c57e354-2662-48bb-8620-d11f6d8ef6d2,deb446fe-0d58-4cef-8c62-b4b89592ee94|deb446fe-...,9999-999,Morning Assessment,2026-03-27 15:30:00,2026-03-27 15:47:37,2026-03-27 15:48:38,1.1.0,3,...,1,2,1,4,0,NA,NA,NA,NA,NA
0,a4314838-bb7f-49d1-9617-b0bdff307e36,107ba4a9-4882-424f-a4ef-5e6d53d690e3,543a00d7-51f5-4dc4-9a8f-a6f422ef82d5|543a00d7-...,9999-999,Afternoon Assessment,2026-03-27 15:45:00,2026-03-27 16:13:05,2026-03-27 16:14:40,1.1.0,NA,...,3,2,1,3,1,2,5,NA,NA,NA


In [13]:
# Write out final flow dataframe: necessary for secondary QC script.

final_out.to_csv(os.path.join(output_path, 'flow_final.csv'), index=False, date_format='%Y-%m-%d %H:%M:%S', na_rep='NA')

In [ ]:
# For a specific participant:
# xxx_ema = final_out[final_out['secret_user_id'] == '9999-999']
# xxx_ema.to_csv(os.path.join(output_path, '9999-999_ema_output_final.csv'), index=False, date_format='%Y-%m-%d %H:%M:%S', na_rep='NA')